# Model: LightGBM

Owner: **Arman**

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

from lightgbm import LGBMRegressor


In [2]:
INCLUDE_RADIATION_DAY_BEFORE = False

MODEL_NAME = (
    "lightgbm"
    if INCLUDE_RADIATION_DAY_BEFORE
    else "lightgbm_no_radiation_day_before"
)

In [3]:
# using the shared train/validation/test sets

DATA_DIR = Path("../../data/NSW")
RESULTS_DIR = Path("../../results")
RESULTS_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "nsw_train.csv", parse_dates=["DATETIME"])
validation = pd.read_csv(DATA_DIR / "nsw_validation.csv", parse_dates=["DATETIME"])
test = pd.read_csv(DATA_DIR / "nsw_test.csv", parse_dates=["DATETIME"])

print(train.shape, validation.shape, test.shape)

(140208, 55) (17520, 55) (17520, 55)


In [4]:
# TEMPERATURE/radiation are same-day actuals so not usable, only the day_before lags are

TARGET = "TOTALDEMAND"
DROP_COLS = ["DATETIME", TARGET, "TEMPERATURE", "radiation", "forecast_closest", "forecast_12hr_prior", "forecast_dayprior"]


if not INCLUDE_RADIATION_DAY_BEFORE:
    DROP_COLS.append("radiation_day_before")

FEATURES = [c for c in train.columns if c not in DROP_COLS]

len(FEATURES)

47

In [5]:
model = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42,
    verbosity=-1,
)

model.fit(train[FEATURES], train[TARGET])

print("Training finished")

Training finished


In [6]:
validation_predictions = model.predict(validation[FEATURES])

validation_rmse = np.sqrt(
    mean_squared_error(validation[TARGET], validation_predictions)
)

print(f"Validation RMSE: {validation_rmse:.2f} MW")

Validation RMSE: 441.29 MW


In [7]:
results = []
best_rmse = float("inf")
best_model = None
best_params = None

for trees in [100, 300]:
    for leaves in [15, 31]:
        candidate = LGBMRegressor(
            n_estimators=trees,
            num_leaves=leaves,
            learning_rate=0.1,
            random_state=42,
            verbosity=-1,
        )

        candidate.fit(train[FEATURES], train[TARGET])
        val_predictions = candidate.predict(validation[FEATURES])

        rmse = np.sqrt(
            mean_squared_error(validation[TARGET], val_predictions)
        )

        results.append({
            "trees": trees,
            "leaves": leaves,
            "validation_rmse": rmse,
        })

        if rmse < best_rmse:
            best_rmse = rmse
            best_model = candidate
            best_params = {"trees": trees, "leaves": leaves}

model = best_model

display(pd.DataFrame(results).sort_values("validation_rmse"))
print("Selected settings:", best_params)

,trees,leaves,validation_rmse
2,300,15,438.828946
3,300,31,439.881030
1,100,31,441.290795
0,100,15,447.335302


Selected settings: {'trees': 300, 'leaves': 15}


In [8]:
predictions = model.predict(test[FEATURES])
print("Number of predictions:", len(predictions))

Number of predictions: 17520


In [9]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    metrics = {"model": model_name, "rmse": rmse, "mae": mae, "mape_pct": mape, "r2": r2}
    print(metrics)
    return metrics


metrics = evaluate(test[TARGET], predictions, MODEL_NAME)

{'model': 'lightgbm_no_radiation_day_before', 'rmse': np.float64(472.4577777822547), 'mae': 311.01191172280147, 'mape_pct': 3.78057823779634, 'r2': 0.8571052106319832}


In [10]:
# saving prediction performance

pd.Series(predictions, index=test["DATETIME"], name=MODEL_NAME).to_csv(RESULTS_DIR / f"{MODEL_NAME}_predictions.csv")

comparison_path = RESULTS_DIR / "model_comparison.csv"
this_run = pd.DataFrame([metrics])

if comparison_path.exists():
    existing = pd.read_csv(comparison_path)
    existing = existing[existing["model"] != MODEL_NAME]
    this_run = pd.concat([existing, this_run], ignore_index=True)

this_run.to_csv(comparison_path, index=False)
this_run

,model,rmse,mae,mape_pct,r2,comments
0,xgboost,470.544565,311.176400,3.791660,0.858260,NaN
1,lightgbm,463.995139,307.403004,3.744587,0.862178,NaN
2,baseline_linear_regression,535.989120,380.215288,4.743792,0.816091,\r\nLinear Regression on the 48 shared feature...
3,prophet,469.383322,345.777006,4.289860,0.843310,NaN
4,random_forest_no_radiation_day_before,502.050043,326.322267,3.962717,0.838644,NaN
5,random_forest,493.696778,322.451412,3.927903,0.843969,NaN
6,aemo_forecast_closest,64.292411,47.711611,0.595688,0.997354,NaN
7,aemo_forecast_12hr_prior,212.913694,155.945260,1.921867,0.970980,NaN
8,aemo_forecast_dayprior,222.460754,161.830989,1.988398,0.968319,NaN
9,lightgbm_no_radiation_day_before,472.457778,311.011912,3.780578,0.857105,NaN


**LightGBM model summary**

Both versions used the same training (2010–2017), validation (2018) and test (2019) periods. Four combinations of tree count and leaf count were compared for each version, with a learning rate of 0.1.
With previous-day radiation included, the model used 48 features and selected 300 trees with 31 leaves. Validation RMSE was 418.75 MW. Test RMSE was 464.00 MW, MAE 307.40 MW, MAPE 3.74% and R² 0.8622.
Without radiation, the model used 47 features and selected 300 trees with 15 leaves. Validation RMSE was 438.83 MW. Test RMSE was 472.46 MW, MAE 311.01 MW, MAPE 3.78% and R² 0.8571.
Including previous-day radiation gave a modest improvement across all four test metrics. This comparison measures its predictive usefulness; it does not establish the effect of solar-panel adoption on electricity demand.